In [65]:
import nltk
import pandas as pd
import requests
import string
from google.colab import userdata
from collections import Counter

In [56]:
# use api to read in bill data
# add environment variable in future to store hashed key in
# max return is 250 but we can add an offset so we can iterate through a certain timeframe to create our dataset

#url = f'https://api.congress.gov/v3/bill?api_key={KEY}'

# things to test
# stemmed vs unstemmed clean vs unclean
# different size word vectors


In [57]:
def load_summaries(key, n=10):
  """
  Take in the number of bills to process (n, int) and output
  a list of summaries and a list of decisions
  """
  # get bills
  params = {'limit':n}
  url_bills = f'https://api.congress.gov/v3/bill?api_key={key}'
  r_bills = requests.get(url_bills, params=params)

  # get bill number and congress number
  numbers = []
  congresses = []
  for bill in r_bills.json()['bills']:
    congresses.append(bill['congress'])
    numbers.append(bill['number'])

  # iterate through bills and get their summaries and max decisions
  sums = []
  decisions = []
  for i in range(len(congresses)):
    url_summaries = f'https://api.congress.gov/v3/bill/{congresses[i]}/hr/{numbers[i]}/summaries?api_key={key}'
    r = requests.get(url_summaries, params=params)
    summaries = r.json()['summaries']
    if len(summaries) > 0:
      sums.append([bill['text'] for bill in summaries][0])
      decisions.append([bill['actionDesc'] for bill in summaries][0])

  return sums, decisions

In [92]:
def stop(txt):
  """
  take in .txt file with words separated by lines and create a list
  of stop words
  """
  with open(txt, 'r') as f:
    stop_words = f.readlines()
  return [word.strip() for word in stop_words]

def clean_txt(texts):
  """
  take in list of strings of text (texts) and remove punctuation, linesplits,
  and stop words (list input)
  """
  clean = []
  # iterate through texts
  for text in texts:

    # remove html type text
    for h in ["<p>", "<strong>", "</p>", "<ul>", "</ul>", "</strong>"]:
      text = text.replace(h, ' ')

    # lower text
    text = text.lower()

    # remove punctuation
    for p in string.punctuation:
      text = text.replace(p, '')

    # remove extra whitespace
    words = text.split()
    for word in words:
      for w in string.whitespace:
        word = word.replace(w, '')

    # rejoin cleaned text
    text = ' '.join(words)
    clean.append(text)

  return clean



def stem(texts):
  """
  Take in list of texts (list (str)) and stem words within texts
  return list of stemmed texts
  """

  ps = nltk.PorterStemmer()
  stemmed = []
  # iterate through all words and stem them using porterstemm from NLTK
  for text in texts:
    for word in text.split():
      text = text.replace(word, ps.stem(word))
    stemmed.append(text)
  return stemmed

def vectorize_txt(txt):
  # insert vectorization here
  pass

In [107]:
def word_bank(texts, stop,  n=200):
  """
  takes in list of texts (list (str)) and finds which words are used the most
  outside of stop words (list (str))
  and returns a list of the top n (int) words
  """
  words = " ".join(texts).split()
  words = [word for word in " ".join(texts).split() if word not in stop]
  counts = Counter(words)
  sort = sorted(counts.items(), key=lambda item: item[1], reverse=True)[:n]
  return [word[0] for word in sort]

In [108]:
#load summaries and decisions
#summaries, decisions = load_summaries(n=5, key=userdata.get('KEY'))

In [109]:
# clean summaries
stopwords = stop('sample_data/stopwords.txt')
summaries_clean = clean_txt(summaries)
summaries_stemmed = stem(summaries_clean)

In [110]:
word_list = word_bank(summaries_clean, stopwords)